In [0]:
CATALOG = 'SALES'
SCHEMA = 'SILVER'
TARGET_TABLE = 'GEOLOCATION_ENRICHED'

In [0]:
geoloc_df = spark.table('SALES.BRONZE.GEOLOCATIONS')
geoloc_df.limit(10).display()

In [0]:
from pyspark.sql.functions import *

geoloc_enriched_df = geoloc_df.select(
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    upper(trim(col("geolocation_city"))).alias("geolocation_city"),
    "file_name",
    "file_path",
    current_timestamp().alias('load_timestamp')
)

display(geoloc_enriched_df.limit(1))

In [0]:
geoloc_enriched_df.createOrReplaceTempView('enriched_geoloc_temp_vw')

In [0]:
%sql
select * from enriched_geoloc_temp_vw

In [0]:
spark.sql(f"""
MERGE INTO {CATALOG}.{SCHEMA}.{TARGET_TABLE} tgt
USING (
  SELECT *
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY  (geolocation_zip_code_prefix, geolocation_lat, geolocation_lng ,geolocation_city)
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_geoloc_temp_vw
  )
  WHERE rn = 1
) src
ON (src.geolocation_zip_code_prefix = tgt.geolocation_zip_code_prefix
and src.geolocation_lat = tgt.geolocation_lat
and src.geolocation_lng = tgt.geolocation_lng
and src.geolocation_city = tgt.geolocation_city)
WHEN MATCHED THEN
  UPDATE SET
    tgt.geolocation_zip_code_prefix = src.geolocation_zip_code_prefix,
    tgt.geolocation_lat = src.geolocation_lat,
    tgt.geolocation_lng = src.geolocation_lng,
    tgt.geolocation_city = src.geolocation_city
WHEN NOT MATCHED THEN
  INSERT (
    geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    geolocation_city,
    origin_file_path,
    load_timestamp
  )
  VALUES (
    src.geolocation_zip_code_prefix,
    src.geolocation_lat,
    src.geolocation_lng,
    src.geolocation_city,
    src.file_path,
    current_timestamp()
  )
""")

     

In [0]:
%sql
  SELECT count(*)
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY  (geolocation_zip_code_prefix, geolocation_lat, geolocation_lng,geolocation_city)
               ORDER BY load_timestamp DESC
           ) rn
    FROM enriched_geoloc_temp_vw
  )
  WHERE rn = 1